In [ ]:
# Importações necessárias para o projeto de machine learning com PyTorch e scikit-learn
import torch  # Biblioteca principal para deep learning
import torch.nn as nn  # Módulos para redes neurais
import torch.optim as optim  # Otimizadores para treinamento
import os  # Para operações do sistema de arquivos
import pandas as pd  # Para manipulação de dados
import matplotlib.pyplot as plt  # Para visualização de dados
import numpy as np    
import random    

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Importações do scikit-learn para pré-processamento e avaliação
from sklearn.model_selection import train_test_split  # Divisão de dados em treino/validação/teste
from sklearn.preprocessing import StandardScaler, LabelEncoder  # Padronização e codificação de labels
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score  # Métricas de avaliação

In [ ]:
from ucimlrepo import fetch_ucirepo  # Biblioteca para acessar datasets do UCI Machine Learning Repository


ucidata = fetch_ucirepo(id=697)  # Busca o dataset pelo ID
X_df = ucidata.data.features  # Features (variáveis independentes) como DataFrame
y_df = ucidata.data.targets  # Targets (variável dependente) como DataFrame

# Encode features e target: transformar variáveis categóricas em numéricas
X = pd.get_dummies(X_df, drop_first=True)  # One-hot encoding para features categóricas
le = LabelEncoder()  # Encoder para labels
y = le.fit_transform(y_df.values.ravel())  # Codifica labels em números inteiros

In [ ]:
# Padronizar os dados (média 0, desvio 1) para melhorar o treinamento da rede neural
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Dividir os dados: 70% treino, 15% validação, 15% teste
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42)  # Primeiro split: treino vs temp

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42)  # Segundo split: val vs teste

# Converter arrays numpy para tensores PyTorch (necessário para o modelo)
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

# Verificar balanceamento das classes nas partições para detectar desequilíbrios
print("Distribuição de classes (train):", torch.bincount(y_train))
print("Distribuição de classes (val):", torch.bincount(y_val))
print("Distribuição de classes (test):", torch.bincount(y_test))

In [ ]:
# Definir a arquitetura da Rede Neural Multi-Camada (MLP)
class MLP(nn.Module):
    def __init__(self, input_dim: int, num_classes: int, dropout_rate: float = 0.3):
        super().__init__()  # Chama o construtor da classe pai   
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),    
            nn.ReLU(),
           # nn.Dropout(dropout_rate),          
            nn.Linear(64, 32),          
            nn.ReLU(),
            nn.Dropout(dropout_rate),                  
            nn.Linear(32, num_classes)   
        )

    def forward(self, x):
        return self.model(x)  # Passa a entrada pela rede

# Instanciar o modelo com as dimensões calculadas
input_dim = X_train.shape[1]  # Número de features de entrada
num_classes = len(le.classes_)  # Número de classes do problema

model = MLP(input_dim, num_classes, dropout_rate=0.2) 

model_path = "mlp_ucirepo.pth"
# try:
#     if os.path.exists(model_path):
#         model.load_state_dict(torch.load(model_path))  
#         model.eval()  
#         print("Modelo carregado com sucesso")
# except RuntimeError as e:
#     print(f"Não foi possível carregar o modelo anterior. Treinando do zero...")

# Calcular pesos predefinidos para contornar o desbalanceamento das classes
class_counts = torch.bincount(y_train)
class_weights = 1. / class_counts.float()
class_weights = class_weights / class_weights.sum()  # Normalizar pesos

# Definir função de perda aplicando os pesos e otimizador (Adam)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Taxa de aprendizado 0.001
#optimizer = optim.SGD(model.parameters(), lr=0.001)


In [ ]:
# Treinamento do modelo (com coleta de loss e acurácia para visualização)
epochs = 100  # Aumentado para 100 épocas para melhor visualização

train_losses = []  # Coleta loss de treino
val_losses = []    # Coleta loss de validação
train_accs = []    # Coleta acurácia de treino
val_accs = []      # Coleta acurácia de validação

for epoch in range(epochs):

    model.train()  # Modo treinamento (ativa dropout, batch norm, etc.)

    outputs = model(X_train)  # Forward pass: calcula saídas
    loss = criterion(outputs, y_train)  # Calcula perda
    _, preds_train = torch.max(outputs, dim=1)
    train_acc = (preds_train == y_train).float().mean().item()

    optimizer.zero_grad()  # Zera gradientes acumulados
    loss.backward()  # Backpropagation: calcula gradientes
    optimizer.step()  # Atualiza pesos

    # Validação: avaliar no conjunto de validação
    model.eval()  # Modo avaliação
    with torch.no_grad():  # Sem cálculo de gradientes
        val_outputs = model(X_val)
        val_loss = criterion(val_outputs, y_val)
        _, preds_val = torch.max(val_outputs, dim=1)
        val_acc = (preds_val == y_val).float().mean().item()

    # Armazena métricas para visualização
    train_losses.append(loss.item())
    val_losses.append(val_loss.item())
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if epoch % 25 == 0:  # Log a cada 10 épocas
        print(f"Epoch {epoch} | Train Loss: {loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")



In [ ]:

# Salvar o modelo treinado para uso futuro
torch.save(model.state_dict(), model_path)
print(f"Modelo salvo em {model_path}")

# Plotar curva de aprendizado: loss e acurácia
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Época')
plt.ylabel('Loss (CrossEntropyLoss)')
plt.title('Curva de Aprendizado - Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Accuracy', linewidth=2)
plt.plot(val_accs, label='Validation Accuracy', linewidth=2)
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.title('Curva de Aprendizado - Acurácia')
plt.legend()

plt.tight_layout()
plt.show()



print(f"Loss Final - Treino: {train_losses[-1]:.4f}, Validação: {val_losses[-1]:.4f}")

In [ ]:
# Avaliação final no conjunto de teste
model.eval()  # Modo avaliação

with torch.no_grad():  # Sem gradientes
    outputs = model(X_test)  # Predições
    _, predicted = torch.max(outputs, 1)  # Classe com maior probabilidade

    # Calcular métricas
    accuracy = (predicted == y_test).sum().item() / len(y_test)  # Acurácia
    precision = precision_score(y_test.numpy(), predicted.numpy(), average="macro")  # Precisão média
    recall = recall_score(y_test.numpy(), predicted.numpy(), average="macro")  # Revocação média

print(f"Acurácia no teste: {accuracy:.2f}")
print(f"Precisão (macro): {precision:.2f}")
print(f"Revocação (macro): {recall:.2f}")

In [ ]:
from sklearn.metrics import f1_score  # Importar F1-score

# Calcular F1-score (média harmônica de precisão e revocação)
f1 = f1_score(y_test.numpy(), predicted.numpy(), average="macro")
print(f"F1-score (macro): {f1:.2f}")

In [ ]:
from sklearn.metrics import roc_curve, auc  # Para curvas ROC
from sklearn.preprocessing import label_binarize  # Binarizar labels para multi-classe

import matplotlib.pyplot as plt  # Para plotar gráficos

# Preparar dados para ROC multi-classe: binarizar labels
y_test_bin = label_binarize(y_test.numpy(), classes=[0, 1, 2])  # Assume 3 classes
n_classes = y_test_bin.shape[1]

# Obter probabilidades previstas (softmax para converter logits em probs)
import torch.nn.functional as F
outputs_prob = F.softmax(outputs, dim=1).detach().numpy()

# Calcular ROC e AUC para cada classe
fpr = dict()  # False Positive Rate
tpr = dict()  # True Positive Rate
roc_auc = dict()  # Área sob a curva
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], outputs_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plotar curvas ROC
plt.figure()
colors = ['blue', 'red', 'green']
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'
             ''.format(i, roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)  # Linha diagonal (classificador aleatório)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# Imprimir AUC para cada classe
for i in range(n_classes):
    print(f"AUC for class {i}: {roc_auc[i]:.2f}")

In [ ]:
# Plotar matriz de confusão para visualizar erros de classificação
cm = confusion_matrix(y_test.numpy(), predicted.numpy())  # Calcula matriz
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)  # Preparar display
disp.plot(cmap="Blues")  # Plotar com mapa de cores azul
plt.title("Matriz de Confusão")
plt.show()

In [ ]:
# Gráficos adicionais de diagnóstico

# 1. Distribuição de Confiança das Predições
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probabilidades máximas (confiança)
max_probs = outputs_prob.max(axis=1)
axes[0].hist(max_probs, bins=30, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Confiança da Predição (probabilidade máxima)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição de Confiança das Predições')
axes[0].axvline(max_probs.mean(), color='red', linestyle='--', label=f'Média: {max_probs.mean():.2f}')
axes[0].legend()

# 2. Acurácia por Classe
from sklearn.metrics import classification_report
report = classification_report(y_test.numpy(), predicted.numpy(), output_dict=True, zero_division=0)
class_names = le.classes_
class_accs = [report[str(i)]['recall'] for i in range(len(class_names))]
axes[1].bar(class_names, class_accs, color=['skyblue', 'lightgreen', 'salmon'], edgecolor='black')
axes[1].set_ylabel('Recall (Sensibilidade)')
axes[1].set_title('Recall por Classe')
axes[1].set_ylim([0, 1.1])
for i, v in enumerate(class_accs):
    axes[1].text(i, v + 0.03, f'{v:.2%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("Relatório de Classificação Detalhado:")
print(classification_report(y_test.numpy(), predicted.numpy(), target_names=class_names, zero_division=0))
